# 04 — Explore visually

Explore the project's three story angles before building the publication
charts, using the shared Pillow templates. Charts render inline via `display()`.

- **A. Domestic** — adjusted vs nominal gross (dumbbell)
- **B. Worldwide** — domestic vs international split (dumbbell)
- **C. Genre** — which genres skew international vs domestic (diverging bars)

> Note: this project renders with the shared **Pillow** factory rather than
> matplotlib. (matplotlib does not run in this project's Python 3.14 venv — a
> known `MarkerStyle` deepcopy recursion during axis-tick rendering — and the
> publication path is Pillow anyway, so matplotlib isn't a dependency here.)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import pandas as pd, duckdb
from src.ingest import load_config
from chart_templates import lollipop, diverging_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: this notebook only reads, so it runs even if another notebook
# kernel has the DuckDB file open (DuckDB is single-writer).
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15').df()
films['label'] = films['title'] + '  (' + films['release_year'].astype(str) + ')'
img_w, img_h, _ = PRESETS['twitter_landscape']
def money(v):
    return f'${abs(v)/1e9:.2f}B' if abs(v) >= 1e9 else f'${abs(v)/1e6:.0f}M'
films[['title','adjusted_gross','nominal_gross','release_year']].head()

## A. Domestic — adjusted vs nominal
The gap between what a film made at the time (gold) and its inflation-adjusted
gross (teal). Domestic (U.S. & Canada) only.

In [ ]:
display(lollipop(
    films, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='Adjusted vs nominal domestic gross - top 15',
    subtitle='Teal = adjusted (2022 $), gold = nominal (release $)',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted', value2_label='Nominal', img_width=img_w, img_height=img_h,
))

## B. Worldwide — domestic vs international
The reason the domestic chart is only half the story: for most top films the
international (rest-of-world) take dwarfs the domestic one. Gold = domestic,
teal = international. Watch **Ne Zha 2** — a Chinese blockbuster with a huge
international total and almost no domestic gross.

In [ ]:
ww = con.execute('''SELECT title, release_year, domestic_gross, foreign_gross, worldwide_gross
    FROM films_worldwide ORDER BY worldwide_gross DESC LIMIT 15''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
display(lollipop(
    ww, category_col='label', value_col='foreign_gross', value2_col='domestic_gross',
    value_fmt=money, title='Worldwide gross: domestic vs international (top 15)',
    subtitle='Teal = international (rest of world), gold = domestic (US & Canada). Nominal $.',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='International', value2_label='Domestic', img_width=img_w, img_height=img_h,
))

## C. Which genres skew international vs domestic
Join the worldwide domestic/foreign gross to TMDB genres and compare each
genre's share of international box office to its share of domestic. The index
= (intl share ÷ domestic share) − 1; positive = over-indexes internationally.
(Each film's gross is attributed to all its genres, so this is a ratio, not a
sum — see SOURCES.md.)

In [ ]:
genre = con.execute('''
    WITH fg AS (SELECT w.domestic_gross, w.foreign_gross, g.genre
                FROM films_worldwide w JOIN film_genres_long g
                  ON g.title=w.title AND g.release_year=w.release_year),
         tot AS (SELECT SUM(domestic_gross) d, SUM(foreign_gross) f FROM fg),
         bygenre AS (SELECT genre, COUNT(*) n, SUM(domestic_gross) dom, SUM(foreign_gross) intl
                     FROM fg GROUP BY genre HAVING COUNT(*)>=10)
    SELECT genre AS category,
           ROUND((intl/(SELECT f FROM tot))/NULLIF(dom/(SELECT d FROM tot),0)-1,3) AS value
    FROM bygenre ORDER BY value DESC''').df()
genre['lbl'] = genre['value'].apply(lambda v: f"{'+' if v>=0 else ''}{v*100:.0f}%")
display(diverging_bars(genre, category_col='category', value_col='value', label_col='lbl',
    title='All genres earn most abroad; some lean more than others',
    subtitle='All 62-68% international; bars = lean vs that norm. Left (e.g. sci-fi) = relatively more domestic, driven by US-heavy franchises.',
    pos_color='#005F73', neg_color='#AE2012', img_width=img_w, img_height=img_h))

## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode,
other notebooks).

In [ ]:
con.close()
print('connection closed')

---
**Next:** `06-viz-social.ipynb` builds the publication versions of these three
charts (full titling/source) and saves them to `outputs/social/`.